# LeetCode 1672 — Richest Customer Wealth bằng Python

Link đề: https://leetcode.com/problems/richest-customer-wealth/

Độ khó: **Easy** · Chủ đề: Array, Matrix

Skeleton LeetCode:

```python
class Solution:
    def maximumWealth(self, accounts: list[list[int]]) -> int:

```

Mục tiêu học:

1. Hiểu **ma trận** (matrix) trong Python là **list của các list**: hàng, cột, `accounts[i][j]`.
2. Viết được hai vòng `for` lồng nhau để duyệt ma trận, và mẫu **"theo dõi max"** (running maximum).
3. Dùng các hàm dựng sẵn `sum`, `max`, `map` để viết lời giải một dòng — và hiểu nó làm gì bên dưới.
4. Phân biệt **tổng theo hàng** (của khách) với **tổng theo cột** (của ngân hàng), làm quen `zip(*matrix)`.
5. Phân tích `O(m · n)` và vì sao không thể làm nhanh hơn.
6. Tự viết lại và giải các biến thể trên ma trận (tìm chỉ số, tổng cột, top-k).

## 1. Bài toán yêu cầu gì?

Cho lưới số nguyên `accounts` kích thước `m × n`: `accounts[i][j]` là số tiền khách hàng thứ `i` có ở ngân hàng thứ `j`.

- **Tài sản (wealth)** của một khách = **tổng** tiền ở mọi ngân hàng = tổng **một hàng**.
- Trả về tài sản của khách **giàu nhất** (giá trị lớn nhất, không phải chỉ số).

| accounts | Tài sản từng khách | Kết quả |
|---|---|---:|
| `[[1,2,3],[3,2,1]]` | `6, 6` | `6` |
| `[[1,5],[7,3],[3,5]]` | `6, 10, 8` | `10` |
| `[[2,8,7],[7,1,3],[1,9,5]]` | `17, 11, 15` | `17` |

Constraints:

- `m == accounts.length`, `n == accounts[i].length`
- `1 <= m, n <= 50`
- `1 <= accounts[i][j] <= 100`

Ví dụ 1 có **hai khách hòa** cùng tài sản `6` — đề chỉ hỏi giá trị, nên trả `6`.

### Đọc ma trận như một bảng

```text
              bank 0  bank 1
customer 0  [   1,      5   ]   -> 6
customer 1  [   7,      3   ]   -> 10  <- giàu nhất
customer 2  [   3,      5   ]   -> 8
```

Mỗi **hàng** là một khách. Ta cần: cộng từng hàng, rồi lấy lớn nhất.

In [1]:
accounts = [[1, 5], [7, 3], [3, 5]]
for i, row in enumerate(accounts):
    print(f"khách {i}: {row} -> tài sản {sum(row)}")

khách 0: [1, 5] -> tài sản 6
khách 1: [7, 3] -> tài sản 10
khách 2: [3, 5] -> tài sản 8


## 2. Nền tảng Python cần biết

### 2.1. Ma trận = list lồng list

- `accounts[i]` là **hàng** thứ `i` (một list).
- `accounts[i][j]` là phần tử ở hàng `i`, cột `j`.
- `len(accounts)` = số hàng `m`; `len(accounts[0])` = số cột `n`.

In [2]:
accounts = [[2, 8, 7], [7, 1, 3], [1, 9, 5]]
print("hàng 1:", accounts[1])
print("accounts[1][2] =", accounts[1][2])
print("m =", len(accounts), "| n =", len(accounts[0]))

hàng 1: [7, 1, 3]
accounts[1][2] = 3
m = 3 | n = 3


### 2.2. Hai vòng `for` lồng nhau

Vòng ngoài chạy qua các hàng, vòng trong chạy qua các phần tử của hàng đó.

In [3]:
accounts = [[1, 2, 3], [3, 2, 1]]
for i in range(len(accounts)):
    for j in range(len(accounts[i])):
        print(f"accounts[{i}][{j}] = {accounts[i][j]}")

accounts[0][0] = 1
accounts[0][1] = 2
accounts[0][2] = 3
accounts[1][0] = 3
accounts[1][1] = 2
accounts[1][2] = 1


Cách viết "Pythonic" hơn: duyệt **trực tiếp** phần tử thay vì chỉ số.

In [4]:
for row in accounts:
    for money in row:
        print(money, end=" ")
    print("| hết một hàng")

1 2 3 | hết một hàng
3 2 1 | hết một hàng


### 2.3. `sum()` và `max()`

- `sum(row)` cộng mọi phần tử của list.
- `max(xs)` trả phần tử lớn nhất.

In [5]:
print(sum([1, 5]), sum([7, 3]), sum([3, 5]))
print(max([6, 10, 8]))

6 10 8
10


### 2.4. Mẫu "theo dõi max" (running maximum)

Giữ biến `best` = lớn nhất **đã thấy cho tới giờ**; gặp giá trị lớn hơn thì cập nhật.

Khởi tạo `best = 0` an toàn vì mọi tài sản `>= 1` (theo constraints). Tổng quát hơn có thể khởi tạo bằng phần tử đầu, hoặc `float("-inf")`.

In [6]:
best = 0
for value in [6, 10, 8]:
    if value > best:
        print(f"{value} > {best} -> cập nhật")
        best = value
    else:
        print(f"{value} <= {best} -> giữ nguyên")
print("best =", best)

6 > 0 -> cập nhật
10 > 6 -> cập nhật
8 <= 10 -> giữ nguyên
best = 10


### 2.5. `map(hàm, list)`

`map(sum, accounts)` áp `sum` lên **từng hàng**, cho ra dãy tổng hàng. `map` trả về một iterator "lười" — dùng `list(...)` để xem.

In [7]:
accounts = [[1, 5], [7, 3], [3, 5]]
print(map(sum, accounts))
print(list(map(sum, accounts)))
print(max(map(sum, accounts)))

[6, 10, 8]
10


## 3. Cách trực giác — cộng tay từng hàng, giữ max

Cách nghĩ của người mới (rất tốt!):

1. `best = 0`.
2. Với mỗi khách: cộng tiền từng ngân hàng vào `wealth`.
3. So `wealth` với `best`, lấy lớn hơn.

Viết hoàn toàn bằng vòng lặp, không dùng `sum`/`max` để thấy rõ từng bước.

In [8]:
def maximum_wealth_loops(accounts: list[list[int]]) -> int:
    best = 0
    for row in accounts:
        wealth = 0
        for money in row:
            wealth += money
        if wealth > best:
            best = wealth
    return best

print(maximum_wealth_loops([[1, 2, 3], [3, 2, 1]]))
print(maximum_wealth_loops([[1, 5], [7, 3], [3, 5]]))
print(maximum_wealth_loops([[2, 8, 7], [7, 1, 3], [1, 9, 5]]))

6
10
17


### Có nút thắt không?

Bài này **không có cách nhanh hơn về Big-O**: muốn biết tổng một hàng, phải đọc **mọi** phần tử của hàng đó (một số chưa đọc có thể làm thay đổi tổng). Vậy mọi lời giải đều phải đọc cả `m · n` ô → `O(m · n)` là tối ưu.

"Cải tiến" ở đây là về **sự rõ ràng** và **tránh lỗi**:

- Dùng hàm dựng sẵn `sum`/`max` → ít dòng, ít chỗ sai.
- Tránh những cách thừa như **lưu cả list tổng rồi sắp xếp** (`O(m log m)` thêm vô ích).

In [9]:
def maximum_wealth_sorted(accounts):
    totals = [sum(row) for row in accounts]
    totals.sort()                    # sắp xếp chỉ để lấy max -> thừa
    return totals[-1]

print(maximum_wealth_sorted([[1, 5], [7, 3], [3, 5]]), "(đúng nhưng sort là thừa)")

10 (đúng nhưng sort là thừa)


## 4. Insight — bài toán là "max của các tổng hàng"

Viết bằng ký hiệu:

```text
đáp án = max over i of ( sum over j of accounts[i][j] )
```

Hai phép gộp lồng nhau: **trong** là `sum` theo hàng, **ngoài** là `max` theo khách. Dịch thẳng sang Python:

```python
max(sum(row) for row in accounts)
```

hoặc

```python
max(map(sum, accounts))
```

Thứ tự gộp quan trọng: `max` **ngoài**, `sum` **trong**. Đảo lại (`sum` của các `max`) là bài khác hẳn.

In [10]:
accounts = [[2, 8, 7], [7, 1, 3], [1, 9, 5]]
print("max của các tổng hàng :", max(sum(r) for r in accounts))
print("tổng của các max hàng :", sum(max(r) for r in accounts), "(bài khác!)")

max của các tổng hàng : 17
tổng của các max hàng : 24 (bài khác!)


## 5. Lời giải chính — `max` của `sum` từng hàng

Giảng từng đoạn:

**Đoạn 1 — tổng từng hàng:** `sum(row)` cho mỗi `row in accounts` (generator, không tạo list trung gian).

**Đoạn 2 — lấy lớn nhất:** bọc bằng `max(...)`.

In [11]:
def maximum_wealth(accounts: list[list[int]]) -> int:
    return max(sum(row) for row in accounts)

print(maximum_wealth([[1, 2, 3], [3, 2, 1]]))
print(maximum_wealth([[1, 5], [7, 3], [3, 5]]))
print(maximum_wealth([[2, 8, 7], [7, 1, 3], [1, 9, 5]]))

6
10
17


`sum(row) for row in accounts` (không có ngoặc vuông) là **generator expression**: sinh từng giá trị khi `max` cần, không tạo list. Nếu viết `[sum(row) for row in accounts]` thì là **list comprehension** — cũng đúng nhưng tốn thêm `O(m)` bộ nhớ.

In [12]:
gen = (sum(row) for row in [[1, 5], [7, 3]])
lst = [sum(row) for row in [[1, 5], [7, 3]]]
print(type(gen).__name__, "|", type(lst).__name__, lst)

generator | list [6, 10]


## 6. TRACE từng bước

Trace ví dụ 3 `[[2,8,7],[7,1,3],[1,9,5]]` với hai vòng lặp, in cả tổng tích lũy trong hàng:

| khách | các bước cộng | wealth | best trước | best sau |
|---:|---|---:|---:|---:|
| 0 | 0+2=2, 2+8=10, 10+7=17 | 17 | 0 | 17 |
| 1 | 0+7=7, 7+1=8, 8+3=11 | 11 | 17 | 17 |
| 2 | 0+1=1, 1+9=10, 10+5=15 | 15 | 17 | 17 |

In [13]:
def trace_wealth(accounts: list[list[int]]) -> int:
    best = 0
    for i, row in enumerate(accounts):
        wealth = 0
        steps = []
        for j, money in enumerate(row):
            wealth += money
            steps.append(f"+{money}={wealth}")
        old = best
        best = max(best, wealth)
        status = "CẬP NHẬT" if best != old else "giữ"
        print(f"khách {i}: {' '.join(steps):<22} wealth={wealth:>3} | best {old:>3} -> {best:>3} ({status})")
    print("Đáp án:", best)
    return best

trace_wealth([[2, 8, 7], [7, 1, 3], [1, 9, 5]])

khách 0: +2=2 +8=10 +7=17       wealth= 17 | best   0 ->  17 (CẬP NHẬT)
khách 1: +7=7 +1=8 +3=11        wealth= 11 | best  17 ->  17 (giữ)
khách 2: +1=1 +9=10 +5=15       wealth= 15 | best  17 ->  17 (giữ)
Đáp án: 17


17

In [14]:
trace_wealth([[1, 2, 3], [3, 2, 1]])   # hai khách hòa: lần 2 không cập nhật vì 6 không > 6

khách 0: +1=1 +2=3 +3=6         wealth=  6 | best   0 ->   6 (CẬP NHẬT)
khách 1: +3=3 +2=5 +1=6         wealth=  6 | best   6 ->   6 (giữ)
Đáp án: 6


6

## 7. Code nộp LeetCode

In [15]:
class Solution:
    def maximumWealth(self, accounts: list[list[int]]) -> int:
        return max(map(sum, accounts))

sol = Solution()
assert sol.maximumWealth([[1, 2, 3], [3, 2, 1]]) == 6
assert sol.maximumWealth([[1, 5], [7, 3], [3, 5]]) == 10
assert sol.maximumWealth([[2, 8, 7], [7, 1, 3], [1, 9, 5]]) == 17
print("3 ví dụ của đề PASS")

3 ví dụ của đề PASS


Bản vòng lặp (khi người phỏng vấn muốn thấy logic, không dùng hàm dựng sẵn):

In [16]:
class SolutionLoops:
    def maximumWealth(self, accounts: list[list[int]]) -> int:
        best = 0
        for row in accounts:
            wealth = 0
            for money in row:
                wealth += money
            best = max(best, wealth)
        return best

assert SolutionLoops().maximumWealth([[1, 5], [7, 3], [3, 5]]) == 10
print("SolutionLoops PASS")

SolutionLoops PASS


## 8. Các lời giải khác

### 8.1. `functools.reduce` — gộp dần một dãy

`reduce(f, xs, init)` tính `f(f(f(init, x0), x1), x2)...`. Có thể dùng nó cho cả `sum` (với `+`) và `max`.

In [17]:
from functools import reduce
import operator

print(reduce(operator.add, [2, 8, 7], 0), "= tổng hàng")

def maximum_wealth_reduce(accounts):
    totals = (reduce(operator.add, row, 0) for row in accounts)
    return reduce(max, totals, 0)

print(maximum_wealth_reduce([[2, 8, 7], [7, 1, 3], [1, 9, 5]]))

17 = tổng hàng
17


`operator.add(a, b)` chính là `a + b` dưới dạng hàm. `reduce(max, totals, 0)` gọi `max(max(max(0, t0), t1), t2)`.

### 8.2. Duyệt theo chỉ số (giống C/Java)

Hữu ích khi cần biết **ai** giàu nhất, hoặc khi chuyển sang ngôn ngữ khác.

In [18]:
def maximum_wealth_index(accounts):
    m, n = len(accounts), len(accounts[0])
    best = 0
    for i in range(m):
        wealth = 0
        for j in range(n):
            wealth += accounts[i][j]
        best = max(best, wealth)
    return best

print(maximum_wealth_index([[1, 5], [7, 3], [3, 5]]))

10


### 8.3. Bảng so sánh các cách

Gọi `m` = số khách, `n` = số ngân hàng.

| Cách | Ý tưởng | Thời gian | Bộ nhớ phụ | Nhận xét |
|---|---|---|---|---|
| Hai vòng lặp tay | cộng từng ô, giữ max | `O(m·n)` | `O(1)` | rõ logic nhất |
| `max(sum(row) for ...)` | generator | `O(m·n)` | `O(1)` | **gọn, nên dùng** |
| `max(map(sum, ...))` | map | `O(m·n)` | `O(1)` | gọn nhất |
| `reduce` | gộp dần | `O(m·n)` | `O(1)` | luyện tư duy hàm |
| Lưu list rồi `sort` | sắp xếp tổng | `O(m·n + m log m)` | `O(m)` | thừa, tránh |

## 9. Vì sao lời giải đúng?

**Invariant của vòng lặp:** sau khi xử lý xong khách thứ `i`, `best` = tài sản lớn nhất trong các khách `0..i`.

- Trước vòng lặp, chưa có khách nào; `best = 0` nhỏ hơn mọi tài sản hợp lệ (mọi ô `>= 1`, nên tài sản `>= 1`).
- Vòng trong cộng đúng mọi ô của hàng `i` → `wealth` = tài sản khách `i`.
- `best = max(best, wealth)` giữ invariant.

Sau khách cuối cùng, `best` là tài sản lớn nhất của **mọi** khách — đúng định nghĩa đề.

Với bản `max(map(sum, accounts))`: `sum` cho đúng tài sản từng khách, `max` cho phần tử lớn nhất của dãy — hai định nghĩa của đề được dịch nguyên văn.

## 10. Độ phức tạp

Gọi `m` = số hàng (khách), `n` = số cột (ngân hàng).

- **Time `O(m · n)`**: mỗi ô được cộng đúng một lần.
- **Space `O(1)`** phụ: chỉ vài biến (`best`, `wealth`); generator/`map` không tạo list.
- Đây là **tối ưu**: phải đọc mọi ô vì bất kỳ ô nào cũng có thể làm thay đổi đáp án.

Với `m, n <= 50`: tối đa 2500 phép cộng.

## 11. Bộ test

1. 3 ví dụ của đề.
2. Biên nhỏ: `1 × 1`.
3. Một khách nhiều ngân hàng; nhiều khách một ngân hàng.
4. Hòa nhau ở max.
5. Biên lớn: `50 × 50` toàn `100` → `5000`.
6. Khách giàu nhất ở **cuối** (kiểm tra không dừng sớm).
7. Đối chiếu mọi cách trên ma trận ngẫu nhiên.

In [19]:
def check(name, actual, expected):
    assert actual == expected, (name, actual, expected)
    print(f"{name:<26} PASS -> {actual}")

f = maximum_wealth
check("đề 1 (hòa)", f([[1, 2, 3], [3, 2, 1]]), 6)
check("đề 2", f([[1, 5], [7, 3], [3, 5]]), 10)
check("đề 3", f([[2, 8, 7], [7, 1, 3], [1, 9, 5]]), 17)
check("1 x 1", f([[42]]), 42)
check("1 khách, 4 ngân hàng", f([[1, 2, 3, 4]]), 10)
check("4 khách, 1 ngân hàng", f([[5], [9], [2], [7]]), 9)
check("50 x 50 toàn 100", f([[100] * 50 for _ in range(50)]), 5000)
check("giàu nhất ở cuối", f([[1, 1], [2, 2], [50, 50]]), 100)

đề 1 (hòa)                 PASS -> 6
đề 2                       PASS -> 10
đề 3                       PASS -> 17
1 x 1                      PASS -> 42
1 khách, 4 ngân hàng       PASS -> 10
4 khách, 1 ngân hàng       PASS -> 9
50 x 50 toàn 100           PASS -> 5000
giàu nhất ở cuối           PASS -> 100


In [20]:
import random

rng = random.Random(1672)
methods = [maximum_wealth_loops, maximum_wealth_sorted, maximum_wealth_reduce,
           maximum_wealth_index, Solution().maximumWealth, SolutionLoops().maximumWealth]
for _ in range(500):
    m, n = rng.randint(1, 8), rng.randint(1, 8)
    acc = [[rng.randint(1, 100) for _ in range(n)] for _ in range(m)]
    ref = maximum_wealth(acc)
    for meth in methods:
        assert meth(acc) == ref
print("6 cách khớp nhau trên 500 ma trận ngẫu nhiên")

6 cách khớp nhau trên 500 ma trận ngẫu nhiên


Chú ý cách tạo ma trận: `[[100] * 50 for _ in range(50)]` — **mỗi hàng là list mới**. Lý do xem lỗi 4 ở dưới.

## 12. Lỗi người mới hay gặp

### Lỗi 1: Cộng theo **cột** thay vì theo **hàng**

Tài sản của **khách** là tổng **hàng**. Tổng cột là tổng tiền của một **ngân hàng**.

In [21]:
accounts = [[1, 5], [7, 3], [3, 5]]
col_sums = [sum(accounts[i][j] for i in range(len(accounts))) for j in range(len(accounts[0]))]
print("sai (max tổng cột):", max(col_sums))
print("đúng (max tổng hàng):", maximum_wealth(accounts))

sai (max tổng cột): 13
đúng (max tổng hàng): 10


### Lỗi 2: Trả về **chỉ số** khách thay vì tài sản

Đề hỏi "the wealth that the richest customer has" — **giá trị**, không phải ai.

In [22]:
def wrong_return_index(accounts):
    totals = [sum(r) for r in accounts]
    return totals.index(max(totals))

print("sai :", wrong_return_index([[1, 5], [7, 3], [3, 5]]))
print("đúng:", maximum_wealth([[1, 5], [7, 3], [3, 5]]))

sai : 1
đúng: 10


### Lỗi 3: Quên reset `wealth` cho mỗi khách

Đặt `wealth = 0` **ngoài** vòng ngoài → tài sản của khách sau bị cộng dồn tài sản khách trước.

In [23]:
def wrong_no_reset(accounts):
    best = 0
    wealth = 0                      # đặt sai chỗ
    for row in accounts:
        for money in row:
            wealth += money
        best = max(best, wealth)
    return best

print("sai :", wrong_no_reset([[1, 5], [7, 3], [3, 5]]), "(= tổng tất cả)")
print("đúng:", maximum_wealth([[1, 5], [7, 3], [3, 5]]))

sai : 24 (= tổng tất cả)
đúng: 10


### Lỗi 4: Tạo ma trận bằng `[[0] * n] * m`

`[[0] * n] * m` tạo `m` **tham chiếu tới cùng một hàng**. Sửa một ô là sửa cả cột! (Không ảnh hưởng input của LeetCode, nhưng hay gây lỗi khi tự tạo test.)

In [24]:
bad = [[0] * 3] * 2
bad[0][0] = 99
print("sai :", bad)

good = [[0] * 3 for _ in range(2)]
good[0][0] = 99
print("đúng:", good)

sai : [[99, 0, 0], [99, 0, 0]]
đúng: [[99, 0, 0], [0, 0, 0]]


### Lỗi 5: Lấy `max` của từng ô rồi cộng

`sum(max(row) ...)` hoặc `max` của mọi ô (`max(max(row) for row in ...)`) là bài khác. Luôn tự hỏi: gộp **trong** là gì, gộp **ngoài** là gì?

In [25]:
acc = [[2, 8, 7], [7, 1, 3], [1, 9, 5]]
print("max một ô     :", max(max(r) for r in acc))
print("đúng (max tổng):", maximum_wealth(acc))

max một ô     : 9
đúng (max tổng): 17


## 13. Mẹo nhận dạng pattern

- "Mỗi hàng là một đối tượng, giá trị = tổng/tích/max của hàng; tìm tốt nhất" → **gộp trong theo hàng, gộp ngoài theo đối tượng**.
- "Tìm lớn nhất / nhỏ nhất" một lượt → **running max/min**, không cần sắp xếp.
- Cần thao tác theo **cột** → `zip(*matrix)` để chuyển vị.
- Ma trận nhỏ, không có cấu trúc đặc biệt → `O(m·n)` đọc hết là tối ưu.

Bài liên quan: 1572 Matrix Diagonal Sum, 1351 Count Negative Numbers in a Sorted Matrix, 867 Transpose Matrix, 2643 Row With Maximum Ones.

## 14. Biến thể

### Biến thể 1: Trả về chỉ số khách giàu nhất (hòa thì chỉ số nhỏ nhất)

`max(range(m), key=...)` trả về **phần tử đầu tiên** đạt max — tự xử lý hòa.

In [26]:
def richest_index(accounts):
    return max(range(len(accounts)), key=lambda i: sum(accounts[i]))

print(richest_index([[1, 5], [7, 3], [3, 5]]))
print(richest_index([[1, 2, 3], [3, 2, 1]]), "(hòa -> chỉ số nhỏ nhất)")

1
0 (hòa -> chỉ số nhỏ nhất)


### Biến thể 2: Ngân hàng giữ nhiều tiền nhất — dùng `zip(*accounts)`

`zip(*accounts)` "trải" các hàng làm đối số cho `zip`, ghép phần tử cùng vị trí → mỗi tuple là **một cột**.

In [27]:
accounts = [[1, 5], [7, 3], [3, 5]]
print("các cột:", list(zip(*accounts)))
print("tổng cột:", [sum(col) for col in zip(*accounts)])
print("ngân hàng giàu nhất có:", max(map(sum, zip(*accounts))))

các cột: [(1, 7, 3), (5, 3, 5)]
tổng cột: [11, 13]
ngân hàng giàu nhất có: 13


### Biến thể 3: Top-k khách giàu nhất

`heapq.nlargest(k, xs)` lấy `k` giá trị lớn nhất trong `O(m log k)`.

In [28]:
import heapq

def top_k_wealth(accounts, k):
    return heapq.nlargest(k, map(sum, accounts))

print(top_k_wealth([[2, 8, 7], [7, 1, 3], [1, 9, 5]], 2))

[17, 15]


### Biến thể 4: Tất cả khách hòa ở mức giàu nhất

In [29]:
def all_richest(accounts):
    totals = [sum(r) for r in accounts]
    best = max(totals)
    return [i for i, t in enumerate(totals) if t == best]

print(all_richest([[1, 2, 3], [3, 2, 1], [1, 1, 1]]))

[0, 1]


## 15. Bài tập tự luyện

Các cell TODO chỉ định nghĩa hàm, không được gọi khi chạy notebook.

### Bài 1 — Dự đoán output

Với `acc = [[3, 3], [1, 6], [4, 1]]`, không chạy code, dự đoán:

1. `[sum(r) for r in acc]`
2. `max(map(sum, acc))`
3. `list(zip(*acc))`
4. `sum(max(r) for r in acc)`

### Bài 2 — Tài sản từng khách

In [30]:
def exercise_wealths(accounts: list[list[int]]) -> list[int]:
    # TODO: trả về list tài sản của từng khách, bằng hai vòng for (không dùng sum)
    pass

# assert exercise_wealths([[1, 5], [7, 3], [3, 5]]) == [6, 10, 8]

### Bài 3 — Viết lại lời giải với running max

In [31]:
def exercise_maximum_wealth(accounts: list[list[int]]) -> int:
    # TODO: best = 0; cộng từng hàng; cập nhật best
    pass

# assert exercise_maximum_wealth([[2, 8, 7], [7, 1, 3], [1, 9, 5]]) == 17

### Bài 4 — Khách **nghèo nhất** và chênh lệch giàu–nghèo

Trả về `max_wealth - min_wealth`.

In [32]:
def exercise_wealth_gap(accounts: list[list[int]]) -> int:
    # TODO: tính min và max của tổng hàng trong MỘT lượt duyệt
    pass

# assert exercise_wealth_gap([[1, 5], [7, 3], [3, 5]]) == 4

### Bài 5 — Biến thể cột: tổng tiền mỗi ngân hàng

In [33]:
def exercise_bank_totals(accounts: list[list[int]]) -> list[int]:
    # TODO: dùng zip(*accounts) hoặc hai vòng for theo cột
    pass

# assert exercise_bank_totals([[1, 5], [7, 3], [3, 5]]) == [11, 13]

### Bài 6 — Mini challenge: Row With Maximum Ones (LeetCode 2643)

Cho ma trận nhị phân `mat`. Trả về `[chỉ_số_hàng, số_lượng_1]` của hàng có nhiều số 1 nhất; hòa thì chọn chỉ số nhỏ nhất.

In [34]:
def exercise_row_max_ones(mat: list[list[int]]) -> list[int]:
    # TODO: running max, chỉ cập nhật khi lớn hơn hẳn
    pass

# assert exercise_row_max_ones([[0, 1], [1, 0]]) == [0, 1]
# assert exercise_row_max_ones([[0, 0, 0], [0, 1, 1]]) == [1, 2]

## 16. Gợi ý 3 mức

**Bài 1**
- Mức 1: tổng hàng: 6, 7, 5.
- Mức 2: các cột: `(3,1,4)` và `(3,6,1)`.
- Mức 3: max từng hàng: 3, 6, 4 → tổng 13.

**Bài 2**
- Mức 1: một list kết quả, mỗi hàng thêm một phần tử.
- Mức 2: `w = 0` cho mỗi hàng rồi cộng.
- Mức 3: `res.append(w)` sau vòng trong.

**Bài 3**
- Mức 1: giống mục 3.
- Mức 2: reset `wealth = 0` **trong** vòng ngoài.
- Mức 3: `best = max(best, wealth)`.

**Bài 4**
- Mức 1: giữ cả `lo` và `hi`.
- Mức 2: khởi tạo `lo = hi = sum(accounts[0])`.
- Mức 3: cập nhật `lo = min(lo, w)`, `hi = max(hi, w)`.

**Bài 5**
- Mức 1: cột `j` gồm `accounts[i][j]` với mọi `i`.
- Mức 2: `zip(*accounts)` cho từng cột.
- Mức 3: `[sum(col) for col in zip(*accounts)]`.

**Bài 6**
- Mức 1: số 1 trong hàng = `sum(row)`.
- Mức 2: `best_i, best_c = 0, -1`.
- Mức 3: chỉ cập nhật khi `c > best_c` (không dùng `>=`) để giữ chỉ số nhỏ nhất.

## 17. Đáp án tham khảo

**Bài 1:** `[6, 7, 5]`; `7`; `[(3, 1, 4), (3, 6, 1)]`; `13`.

In [35]:
def answer_wealths(accounts):
    res = []
    for row in accounts:
        w = 0
        for money in row:
            w += money
        res.append(w)
    return res

def answer_maximum_wealth(accounts):
    best = 0
    for row in accounts:
        wealth = 0
        for money in row:
            wealth += money
        best = max(best, wealth)
    return best

def answer_wealth_gap(accounts):
    lo = hi = sum(accounts[0])
    for row in accounts[1:]:
        w = sum(row)
        lo, hi = min(lo, w), max(hi, w)
    return hi - lo

def answer_bank_totals(accounts):
    return [sum(col) for col in zip(*accounts)]

def answer_row_max_ones(mat):
    best_i, best_c = 0, -1
    for i, row in enumerate(mat):
        c = sum(row)
        if c > best_c:
            best_i, best_c = i, c
    return [best_i, best_c]

acc = [[3, 3], [1, 6], [4, 1]]
assert [sum(r) for r in acc] == [6, 7, 5] and max(map(sum, acc)) == 7          # Bài 1
assert list(zip(*acc)) == [(3, 1, 4), (3, 6, 1)] and sum(max(r) for r in acc) == 13
assert answer_wealths([[1, 5], [7, 3], [3, 5]]) == [6, 10, 8]
assert answer_maximum_wealth([[2, 8, 7], [7, 1, 3], [1, 9, 5]]) == 17
assert answer_wealth_gap([[1, 5], [7, 3], [3, 5]]) == 4
assert answer_bank_totals([[1, 5], [7, 3], [3, 5]]) == [11, 13]
assert answer_row_max_ones([[0, 1], [1, 0]]) == [0, 1]
assert answer_row_max_ones([[0, 0, 0], [0, 1, 1]]) == [1, 2]
print("Tất cả đáp án PASS")

Tất cả đáp án PASS


## 18. Template nhớ nhanh + checklist submit

```python
class Solution:
    def maximumWealth(self, accounts: list[list[int]]) -> int:
        return max(map(sum, accounts))     # max (ngoài) của sum từng hàng (trong)
```

Bản vòng lặp:

```python
best = 0
for row in accounts:
    wealth = 0              # reset cho mỗi khách
    for money in row:
        wealth += money
    best = max(best, wealth)
return best
```

Câu thần chú: **"Khách = hàng. Cộng hàng, lấy max."**

Checklist trước khi submit:

- [ ] Cộng theo **hàng** (khách), không theo cột (ngân hàng)?
- [ ] Trả về **giá trị** tài sản, không phải chỉ số?
- [ ] `wealth` được reset cho mỗi khách?
- [ ] Không sắp xếp thừa chỉ để lấy max?
- [ ] Đã thử ví dụ hòa `[[1,2,3],[3,2,1]]` → `6`?